<a href="https://colab.research.google.com/github/kouameyannick/Nanan_AI/blob/main/ASR_Baoule_ASR_Traduction_complet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmark Baoulé : ASR et traduction Baoulé ↔ français

Ce notebook regroupe les tests présents dans le notebook original :
1. **ASR** : audio baoulé → transcription avec Omnilingual ASR.
2. **Griot-MT** : traduction baoulé → français.
3. **NLLB Baoulé v1** : comparaison sur la même phrase.
4. **Griot-MT** : traduction français → baoulé.

**Important :** exécute les cellules dans l’ordre. Les deux environnements sont séparés volontairement :
- Omnilingual ASR utilise Python 3.11 dans `/content/venv311`.
- Les modèles de traduction utilisent l’environnement standard de Colab.

Les modèles doivent être téléchargés au premier lancement. Leur chargement peut prendre du temps et nécessite une connexion Internet. Les sorties des modèles sont des résultats à évaluer, pas une garantie de traduction exacte.

## 1. Vérifier l'environnement Colab

Dans **Exécution → Modifier le type d'exécution**, choisis un GPU si disponible. Cette cellule ne modifie aucun paquet.

In [1]:
import sys, subprocess
print("Python Colab :", sys.version.split()[0])
gpu = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "GPU non détecté. Les modèles peuvent être très lents sur CPU.")

Python Colab : 3.13.15
Mon Oct 19 19:04:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------

## 2. Préparer Python 3.11 pour Omnilingual ASR

Cette cellule crée l'environnement uniquement s'il n'existe pas encore. Elle ne supprime pas un environnement existant.

In [2]:
from pathlib import Path
import subprocess, sys

venv_python = Path("/content/venv311/bin/python")
if not venv_python.exists():
    subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq python3.11 python3.11-dev python3.11-venv libsndfile1"], check=True)
    subprocess.run(["python3.11", "-m", "venv", "/content/venv311"], check=True)
    subprocess.run([str(venv_python), "-m", "pip", "install", "--upgrade", "pip"], check=True)
else:
    print("Environnement /content/venv311 déjà présent : il est conservé.")

print(subprocess.run([str(venv_python), "--version"], capture_output=True, text=True, check=True).stdout.strip())

Python 3.11.17


## 3. Installer Omnilingual ASR (une seule fois par session Colab)

Cette installation est indépendante des bibliothèques de traduction. PyTorch 2.8.0 avec CUDA 12.8 reprend la configuration utilisée dans le notebook original. Si une installation échoue, lis l'erreur avant de relancer ou de changer les versions.

In [3]:
from pathlib import Path
import subprocess

venv_python = "/content/venv311/bin/python"

def run_cmd(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout[-5000:])
    if result.returncode != 0:
        print("\n--- ERREUR ---\n", result.stderr[-10000:])
        raise RuntimeError("La commande a échoué. Lis l'erreur ci-dessus avant de continuer.")
    return result

check_asr = subprocess.run(
    [venv_python, "-c", "import torch, torchaudio, omnilingual_asr; print('ASR_DEPS_OK')"],
    capture_output=True, text=True
)
if check_asr.returncode == 0:
    print("Les dépendances ASR sont déjà installées : aucune réinstallation.")
else:
    run_cmd([venv_python, "-m", "pip", "install", "torch==2.8.0", "torchaudio==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu128"])
    run_cmd([venv_python, "-m", "pip", "install", "omnilingual-asr"])

    # Correction ciblée de la combinaison NumPy/PyArrow signalée dans les tests précédents.
    run_cmd([venv_python, "-m", "pip", "install", "--force-reinstall", "--no-deps", "pyarrow==18.1.0"])

run_cmd([venv_python, "-c", "import sys, torch, torchaudio, omnilingual_asr; print('Python :', sys.version.split()[0]); print('PyTorch :', torch.__version__); print('TorchAudio :', torchaudio.__version__); print('CUDA disponible :', torch.cuda.is_available()); print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'); print('Omnilingual ASR : import OK')"])

━━━━━━━━━━━━━━ 594.3/594.3 MB 141.9 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 73.4 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 118.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 37.3 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 132.0 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 35.4 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 37.4 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 206.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 332.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.2/889.2 MB 11.5 MB/s  0:00:29
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 15.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 66.0 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 69.5 MB/s  0:00:00
   ━━━━━━━

CompletedProcess(args=['/content/venv311/bin/python', '-c', "import sys, torch, torchaudio, omnilingual_asr; print('Python :', sys.version.split()[0]); print('PyTorch :', torch.__version__); print('TorchAudio :', torchaudio.__version__); print('CUDA disponible :', torch.cuda.is_available()); print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'); print('Omnilingual ASR : import OK')"], returncode=0, stdout='Python : 3.11.17\nPyTorch : 2.8.0+cu128\nTorchAudio : 2.8.0+cu128\nCUDA disponible : True\nGPU : Tesla T4\nOmnilingual ASR : import OK\n', stderr='')

## 4. Importer un court audio en baoulé

Choisis un fichier audio court et clair. Le chemin réel du fichier importé sera utilisé automatiquement dans le test.

In [5]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise ValueError("Aucun fichier audio n'a été importé.")
audio_path = str(Path(next(iter(uploaded))).resolve())
print("Fichier audio :", audio_path)
print("Taille (Mo) :", round(Path(audio_path).stat().st_size / (1024 * 1024), 2))

Saving common_voice_bci_41240245.mp3 to common_voice_bci_41240245.mp3
Fichier audio : /content/common_voice_bci_41240245.mp3
Taille (Mo) : 0.03


## 5. Lancer la reconnaissance vocale (ASR)

Modèle de départ : `omniASR_CTC_300M_v2`. Code langue utilisé dans le notebook original : `bci_Latn`. La première exécution télécharge les poids du modèle.

In [6]:
import subprocess, textwrap
from pathlib import Path

MODEL_CARD = "omniASR_CTC_300M_v2"
LANG_CODE = "bci_Latn"

asr_script = r'''
import sys, torch
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

audio_path, model_card, lang_code = sys.argv[1:4]
print("Chargement du modèle :", model_card)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

pipeline = ASRInferencePipeline(model_card=model_card)
print("Modèle chargé. Transcription en cours...")
result = pipeline.transcribe([audio_path], lang=[lang_code], batch_size=1)
print("\n===== TRANSCRIPTION BAOULÉ =====")
print(result[0])
'''
script_path = "/content/run_asr_baoule.py"
Path(script_path).write_text(asr_script, encoding="utf-8")

run = subprocess.run(
    ["/content/venv311/bin/python", script_path, audio_path, MODEL_CARD, LANG_CODE],
    text=True, capture_output=True
)
print(run.stdout)
if run.returncode != 0:
    print("\n===== ERREUR ASR =====\n", run.stderr[-12000:])
    raise RuntimeError("La transcription ASR a échoué. Ne réinstalle pas tous les paquets : utilise le message d'erreur affiché pour corriger la cause précise.")

Chargement du modèle : omniASR_CTC_300M_v2
CUDA disponible : True
GPU : Tesla T4
Modèle chargé. Transcription en cours...

===== TRANSCRIPTION BAOULÉ =====
ɔ wlatralɛ bleklanman kpa titi



## 6. Installer/vérifier les dépendances de traduction

Le notebook original a utilisé ces versions après correction de l'erreur PEFT/Transformers. Cette cellule vérifie d'abord les imports et n'installe les paquets que s'ils manquent ou si les versions ne correspondent pas.

In [7]:
import importlib.metadata as metadata
import subprocess, sys

required_versions = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "accelerate": "1.14.0",
    "sentencepiece": "0.2.2",
}
needs_install = []
for package, expected in required_versions.items():
    try:
        current = metadata.version(package)
        print(f"{package}: {current} (attendu {expected})")
        if current != expected:
            needs_install.append(f"{package}=={expected}")
    except metadata.PackageNotFoundError:
        needs_install.append(f"{package}=={expected}")

try:
    import torchao
    print("torchao :", metadata.version("torchao"))
except Exception:
    needs_install.append("torchao>=0.17.0")

if needs_install:
    print("Installation/correction des dépendances :", needs_install)
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", *needs_install,
                    "safetensors>=0.4.3", "huggingface_hub>=0.34.0"], check=True)
    print("Installation terminée. Si Colab demande un redémarrage du runtime, redémarre puis reprends à cette section.")
else:
    print("Les versions principales attendues sont déjà présentes.")

transformers: 5.20.1 (attendu 5.16.1)
peft: 0.21.2 (attendu 0.20.0)
accelerate: 1.15.0 (attendu 1.14.0)
sentencepiece: 0.2.2 (attendu 0.2.2)
torchao : 0.10.0
Installation/correction des dépendances : ['transformers==5.16.1', 'peft==0.20.0', 'accelerate==1.14.0']
Installation terminée. Si Colab demande un redémarrage du runtime, redémarre puis reprends à cette section.


## 7. Charger Griot-MT avec l'adaptateur Baoulé

Griot-MT est utilisé ici pour les deux sens de traduction. Le chargement peut être long. Si les dépendances viennent d'être installées et que les imports échouent, redémarre le runtime puis reprends à cette cellule.

In [1]:
!pip install -U "torchao>=0.17.0"

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

GRIOT_MODEL_ID = "bivariant/griot-mt"
print("Chargement du tokenizer Griot-MT...")
tokenizer = AutoTokenizer.from_pretrained(GRIOT_MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Chargement du modèle de base...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    GRIOT_MODEL_ID,
    torch_dtype=dtype
).to(device)

print("Chargement de l'adaptateur Baoulé...")
model = PeftModel.from_pretrained(
    base_model, GRIOT_MODEL_ID, subfolder="adapters/bci"
)
model.eval()
print("Griot-MT prêt. Appareil :", device)

Chargement du tokenizer Griot-MT...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Chargement du modèle de base...


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Chargement de l'adaptateur Baoulé...


adapters/bci/adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 94.5MB            

adapters/bci/adapter_model.safetensors: downloading bytes:           |  0.00B            

Griot-MT prêt. Appareil : cuda


## 8. Test Griot-MT : Baoulé → français

Phrase de test reprise du notebook original. Utilise exactement la même phrase pour comparer avec NLLB.

In [3]:
FRENCH_TOKEN = "fra_Latn"
BAOULE_TOKEN = "bci_Latn"

def traduire_griot(texte, langue_source, langue_cible, max_new_tokens=128):
    tokenizer.src_lang = langue_source
    inputs = tokenizer(texte, return_tensors="pt", truncation=True).to(device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(langue_cible),
            num_beams=4,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

texte_baoule = "I sin’n, ɔ mɛnnin i wun’n wie. Yɛ i kusu dili ɔ."
traduction_griot_fr = traduire_griot(texte_baoule, BAOULE_TOKEN, FRENCH_TOKEN)

print("Baoulé :", texte_baoule)
print("Griot-MT → français :", traduction_griot_fr)

Baoulé : I sin’n, ɔ mɛnnin i wun’n wie. Yɛ i kusu dili ɔ.
Griot-MT → français : Elle en donna ensuite à son mari, qui en mangea aussi.


## 9. Charger NLLB Baoulé v1

Ce modèle est testé séparément sur la même phrase baoulé. Il peut nécessiter beaucoup de mémoire GPU. On conserve le type de précision adapté à l'appareil.

In [4]:
NLLB_MODEL_ID = "GaindeNdiaye/nllb-baoule-v1"
print("Chargement du tokenizer NLLB...")
tokenizer_nllb = AutoTokenizer.from_pretrained(NLLB_MODEL_ID)

device_nllb = "cuda" if torch.cuda.is_available() else "cpu"
dtype_nllb = torch.float16 if device_nllb == "cuda" else torch.float32
print("Chargement du modèle NLLB...")
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_ID,
    torch_dtype=dtype_nllb
).to(device_nllb)
model_nllb.eval()
print("NLLB prêt. Appareil :", device_nllb)

Chargement du tokenizer NLLB...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

NLLB prêt. Appareil : cuda


## 10. Test NLLB : Baoulé → français et comparaison

La comparaison porte sur une seule phrase : elle permet de vérifier que les deux modèles produisent une sortie, mais elle ne suffit pas à conclure qu'un modèle est globalement meilleur.

In [5]:
tokenizer_nllb.src_lang = BAOULE_TOKEN
inputs_nllb = tokenizer_nllb(
    texte_baoule, return_tensors="pt", truncation=True
).to(device_nllb)

with torch.inference_mode():
    output_nllb = model_nllb.generate(
        **inputs_nllb,
        forced_bos_token_id=tokenizer_nllb.convert_tokens_to_ids(FRENCH_TOKEN),
        max_new_tokens=128,
        num_beams=4
    )

traduction_nllb_fr = tokenizer_nllb.decode(output_nllb[0], skip_special_tokens=True)

print("Phrase source :", texte_baoule)
print("\nGriot-MT :", traduction_griot_fr)
print("NLLB Baoulé v1 :", traduction_nllb_fr)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Phrase source : I sin’n, ɔ mɛnnin i wun’n wie. Yɛ i kusu dili ɔ.

Griot-MT : Elle en donna ensuite à son mari, qui en mangea aussi.
NLLB Baoulé v1 : Elle en donna aussi à son mari, qui en mangea.


## 11. Test Griot-MT : français → baoulé

Phrase de test reprise du notebook original, avec correction de la faute de frappe « hommemange » → « homme mange ». Tu peux remplacer cette phrase par une phrase française plus courte pour commencer.

In [6]:
phrase_francaise = (
    "J’ai appris qu’il n’y a rien de mieux pour les humains que de se réjouir "
    "et de faire le bien durant leur vie, et aussi que tout homme mange, "
    "boive et voie le bien pour tout son dur travail."
)
traduction_fr_baoule = traduire_griot(
    phrase_francaise, FRENCH_TOKEN, BAOULE_TOKEN, max_new_tokens=160
)

print("Français :", phrase_francaise)
print("Griot-MT → baoulé :", traduction_fr_baoule)

Français : J’ai appris qu’il n’y a rien de mieux pour les humains que de se réjouir et de faire le bien durant leur vie, et aussi que tout homme mange, boive et voie le bien pour tout son dur travail.
Griot-MT → baoulé : N wunnin i wlɛ kɛ like fi nun-man lɛ b'ɔ ti kpa tra sran'm be klun jɔlɛ dilɛ'n, ɔ nin like kpa nga be yo i cɛn nga be tɛ o lɛ'n. I kunngba'n yɛlɛ kɛ sran kun `di like, yɛ ɔ `nɔn nzan, yɛ i junman kekle'n kwlaa man ɔ nyan nyrɛnnɛn.


## 12. Enregistrer les résultats du benchmark

Ce tableau rassemble les sorties produites pendant cette session. Il ne calcule pas de score automatique, car il faudrait des transcriptions/traductions de référence vérifiées par un locuteur baoulé.

In [7]:
import pandas as pd
from datetime import datetime

resultats = [
    {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "tâche": "ASR audio → texte baoulé",
     "modèle": MODEL_CARD, "entrée": Path(audio_path).name,
     "sortie": run.stdout.split("===== TRANSCRIPTION BAOULÉ =====")[-1].strip()},
    {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "tâche": "Baoulé → français",
     "modèle": "Griot-MT", "entrée": texte_baoule, "sortie": traduction_griot_fr},
    {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "tâche": "Baoulé → français",
     "modèle": "NLLB Baoulé v1", "entrée": texte_baoule, "sortie": traduction_nllb_fr},
    {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "tâche": "Français → baoulé",
     "modèle": "Griot-MT", "entrée": phrase_francaise, "sortie": traduction_fr_baoule},
]
df_resultats = pd.DataFrame(resultats)
display(df_resultats)
output_csv = "/content/benchmark_baoule.csv"
df_resultats.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("Résultats enregistrés dans :", output_csv)

NameError: name 'MODEL_CARD' is not defined

## 13. Comment interpréter les résultats

- Vérifie la transcription ASR en la comparant à ce qui est réellement prononcé.
- Pour la traduction, compare le sens, les noms propres, les omissions et les ajouts.
- Une seule phrase ne suffit pas pour classer les modèles : teste plusieurs phrases baoulé et fais vérifier les références par un locuteur compétent.
- Si une cellule échoue, conserve le traceback complet et corrige cette cellule précise. Ne réinstalle pas toutes les bibliothèques au hasard.